![Health-Informatics: A Python Tutorial](../../Image/chapter-banner.png)


# Module 4: Health Data Standards and Formats



**Health Informatics in Python** · Part I: Foundations · Module 4 of 16

---

Data that can't move between systems isn't much use. **Interoperability standards**
define how a patient record is packaged so a receiving system can understand it.
This module walks the ladder from flat files up through **HL7 v2**, **CDA**, and
**FHIR**, parsing each in Python.


## Learning objectives

By the end of this module you will be able to:

1. Place the main health-data formats on a spectrum from **flat files → HL7 v2 →
   CDA → FHIR** and say when each is used.
2. **Parse an HL7 v2** message and pull structured values out of it.
3. **Parse a CDA** document (XML) with the standard library.
4. Construct and serialize **FHIR resources** with `fhir.resources`.
5. Represent the **same patient fact** across all three modern standards.


## Dataset

We use small, hand-built sample messages — one **HL7 v2** lab result, one **CDA**
fragment, and FHIR resources we construct in code. They all describe the same
synthetic patient (Ava Khan, A1c 7.2%) so you can watch one fact change costume.


## 4.1 The interoperability ladder

| Format | Era / use | Shape | Strength |
|---|---|---|---|
| **Flat file (CSV)** | Bulk extracts, research | Rows & columns | Simple, universal |
| **HL7 v2** | Real-time messaging (still dominant in hospitals) | Pipe-delimited segments | Ubiquitous, battle-tested |
| **CDA** | Clinical documents (discharge summaries) | XML | Human- + machine-readable docs |
| **FHIR** | Modern APIs, apps, exchange | JSON/XML resources + REST | Web-native, granular, extensible |

Reality check: **HL7 v2 still carries the majority of live hospital messaging**,
while **FHIR** is where new development happens. You will meet both for years.


## 4.2 Flat files - the baseline

In [1]:
# A FLAT FILE is the simplest health-data format: rows and columns, like a spreadsheet.
# Easy to parse, but it carries almost no standard meaning - nothing here says
# "this is a LOINC lab" or "this patient matches that hospital's medical-record ID."
#
# io.StringIO(csv) pretends a string is a file, so pd.read_csv can read it
# without writing anything to disk.
import pandas as pd
import io

csv = '''patient_id,observation,value,unit
P1001,Hemoglobin A1c,7.2,%
P1001,Systolic blood pressure,148,mmHg'''

flat = pd.read_csv(io.StringIO(csv))
print("Flat file — trivially parsed, but no standard semantics or metadata:")
print(flat.to_string(index=False))


Flat file — trivially parsed, but no standard semantics or metadata:
patient_id             observation  value unit
     P1001          Hemoglobin A1c    7.2    %
     P1001 Systolic blood pressure  148.0 mmHg


## 4.3 HL7 v2 - segments and fields

An HL7 v2 message is a set of **segments** (lines), each split into **fields** by
`|`, with `^` separating components. `MSH` is the header, `PID` the patient,
`OBX` an observation. It looks archaic but it's everywhere.


In [2]:
# An HL7 v2 message is a text blob hospitals still use for real-time lab results,
# ADT (admit/discharge/transfer) events, and orders. It looks cryptic. Here is
# how to read it:
#
#   Each LINE is a SEGMENT (MSH, PID, OBR, OBX, ...)
#   Fields inside a segment are split by the pipe character |
#   Pieces inside a field are split by ^
#
#   MSH = message header (who sent it, what kind of message, version)
#   PID = patient identification (ID, name, date of birth, sex, address)
#   OBR = order/observation request (which test was ordered)
#   OBX = observation result (the actual lab/vital value)
#
# This sample says: Ava Khan (P1001) has A1c 7.2% (high) and SBP 148 mmHg (high).
hl7_msg = '''MSH|^~\&|LAB|CENTRAL_HOSP|EHR|CENTRAL_HOSP|20260115093000||ORU^R01|MSG00001|P|2.5
PID|1||P1001^^^HOSP^MR||Khan^Ava^L||19680312|F|||12 Elm St^^Ithaca^NY^14850
OBR|1|||4548-4^Hemoglobin A1c^LN
OBX|1|NM|4548-4^Hemoglobin A1c^LN||7.2|%|4.0-5.6|H|||F
OBX|2|NM|8480-6^Systolic blood pressure^LN||148|mmHg|90-120|H|||F'''
print(hl7_msg)


MSH|^~\&|LAB|CENTRAL_HOSP|EHR|CENTRAL_HOSP|20260115093000||ORU^R01|MSG00001|P|2.5
PID|1||P1001^^^HOSP^MR||Khan^Ava^L||19680312|F|||12 Elm St^^Ithaca^NY^14850
OBR|1|||4548-4^Hemoglobin A1c^LN
OBX|1|NM|4548-4^Hemoglobin A1c^LN||7.2|%|4.0-5.6|H|||F
OBX|2|NM|8480-6^Systolic blood pressure^LN||148|mmHg|90-120|H|||F


In [4]:
# The `hl7` Python library knows how to split that pipe-delimited text into
# segments and fields so we don't have to do it by hand.
import hl7

# Real HL7 v2 messages separate segments with a carriage return (\r), not a
# normal newline (\n). Our sample used newlines so it would display nicely,
# so we convert them before parsing.
h = hl7.parse(hl7_msg.replace("\n", "\r"))

# Each segment's first field is its 3-letter name (MSH, PID, OBR, OBX, ...)
print("Segments present:", [str(seg[0]) for seg in h])

# h.segment("PID") returns the patient-identification segment.
# Fields are numbered from 1 in the spec; the library uses the same numbering.
#   PID-5 = patient name (Khan^Ava^L  →  family^given^middle)
#   PID-7 = date of birth (YYYYMMDD)
#   PID-8 = sex
pid = h.segment("PID")
print("\nPatient name (PID-5):", pid[5])
print("Patient DOB  (PID-7):", pid[7])
print("Patient sex  (PID-8):", pid[8])


Segments present: ['MSH', 'PID', 'OBR', 'OBX', 'OBX']

Patient name (PID-5): Khan^Ava^L
Patient DOB  (PID-7): 19680312
Patient sex  (PID-8): F


### Milestone 1 - extract all OBX results into a table

In [5]:
# Walk every segment in the parsed message. When we find an OBX (a result),
# pull out the useful fields and collect them into a pandas table.
#
# Inside an OBX, the important field numbers are:
#   OBX-3 = observation identifier  (LOINC^name^coding-system, split by ^)
#   OBX-5 = the value               (e.g. 7.2)
#   OBX-6 = the unit                (e.g. %)
#   OBX-8 = abnormal flag           (H = high, L = low, N = normal)
rows = []
for seg in h:
    if str(seg[0]) == "OBX":
        code_field = str(seg[3])  # e.g. "4548-4^Hemoglobin A1c^LN"
        # split on ^ : piece 0 is the LOINC code, piece 1 is the display name
        loinc, name = code_field.split("^")[0], code_field.split("^")[1]
        rows.append({
            "loinc": loinc,
            "observation": name,
            "value": str(seg[5]),
            "unit": str(seg[6]),
            "flag": str(seg[8]),
        })

obx = pd.DataFrame(rows)
print("Structured results parsed from HL7 v2 OBX segments:")
print(obx.to_string(index=False))


Structured results parsed from HL7 v2 OBX segments:
 loinc             observation value unit flag
4548-4          Hemoglobin A1c   7.2    %    H
8480-6 Systolic blood pressure   148 mmHg    H


## 4.4 CDA - clinical documents in XML

A **CDA** (Clinical Document Architecture) document is XML with a fixed header and
structured body. It represents whole documents — discharge summaries, referral
notes. We parse a minimal fragment with the standard library.


In [6]:
# CDA (Clinical Document Architecture) is XML: nested tags instead of pipes.
# It is used for whole documents (discharge summaries, referral notes) that a
# human can also open and read.
#
# The important nested pieces in this fragment:
#   <id extension="P1001"/>              → patient ID
#   <given>Ava</given><family>Khan</family>  → name
#   <administrativeGenderCode code="F"/> → sex
#   <birthTime value="19680312"/>        → date of birth
#   <code code="4548-4" .../>            → LOINC for A1c
#   <value unit="%" value="7.2"/>        → the result
#
# xmlns="urn:hl7-org:v3" is the CDA namespace — we will need it when we search.
cda = """<?xml version="1.0"?>
<ClinicalDocument xmlns="urn:hl7-org:v3">
  <recordTarget>
    <patientRole>
      <id extension="P1001"/>
      <patient>
        <name><given>Ava</given><family>Khan</family></name>
        <administrativeGenderCode code="F"/>
        <birthTime value="19680312"/>
      </patient>
    </patientRole>
  </recordTarget>
  <component>
    <observation>
      <code code="4548-4" codeSystemName="LOINC" displayName="Hemoglobin A1c"/>
      <value unit="%" value="7.2"/>
    </observation>
  </component>
</ClinicalDocument>"""
print(cda[:180], "...")  # print just the start so you see the shape without a wall of XML


<?xml version="1.0"?>
<ClinicalDocument xmlns="urn:hl7-org:v3">
  <recordTarget>
    <patientRole>
      <id extension="P1001"/>
      <patient>
        <name><given>Ava</given><fa ...


In [7]:
# xml.etree.ElementTree is in the Python standard library — no extra install.
# It turns the XML string into a tree of nodes we can search.
import xml.etree.ElementTree as ET

# CDA tags live in a namespace. find() needs that namespace or it will see
# nothing. We give it a short nickname "v3" so paths can say v3:patient, etc.
ns = {"v3": "urn:hl7-org:v3"}

root = ET.fromstring(cda)  # parse the XML string into a tree; root is <ClinicalDocument>

# .// means "search anywhere below root". Paths walk the nested tags.
# .text  = the words between a tag pair, e.g. <given>Ava</given> → "Ava"
# .get("code") = an attribute on the tag, e.g. code="F" → "F"
given  = root.find(".//v3:patient/v3:name/v3:given", ns).text
family = root.find(".//v3:patient/v3:name/v3:family", ns).text
sex    = root.find(".//v3:administrativeGenderCode", ns).get("code")
dob    = root.find(".//v3:birthTime", ns).get("value")
obs    = root.find(".//v3:observation/v3:code", ns).get("displayName")
val    = root.find(".//v3:observation/v3:value", ns)

print(f"Patient : {given} {family}  (sex={sex}, dob={dob})")
print(f"Observation: {obs} = {val.get('value')} {val.get('unit')}")


Patient : Ava Khan  (sex=F, dob=19680312)
Observation: Hemoglobin A1c = 7.2 %


## 4.5 FHIR - resources and REST

**FHIR** breaks the record into modular **resources** (`Patient`, `Observation`,
`Condition`, ...) exchanged as JSON/XML over a REST API. It is the modern standard
and the one you'll build against most. We construct and validate resources with
`fhir.resources` (which enforces the FHIR schema via pydantic).


In [8]:
# FHIR breaks the record into modular RESOURCES. A Patient resource is a
# JSON (or XML) object with a defined schema: id, name, gender, birthDate, ...
#
# fhir.resources is a Python library that:
#   1. lets us build those objects with keyword arguments
#   2. checks they match the FHIR schema (via pydantic)
#   3. can dump them back to JSON for sending over an API
from fhir.resources.patient import Patient

# Build one Patient resource for Ava Khan — the same person as in the HL7/CDA samples.
# name is a list because a person can have several names (official, maiden, nickname).
patient = Patient(
    id="P1001",
    name=[{"family": "Khan", "given": ["Ava"], "use": "official"}],
    gender="female",
    birthDate="1968-03-12",
)

# model_dump_json() serializes the object to FHIR JSON (indent=2 makes it readable)
print(patient.model_dump_json(indent=2))


{
  "resourceType": "Patient",
  "id": "P1001",
  "name": [
    {
      "use": "official",
      "family": "Khan",
      "given": [
        "Ava"
      ]
    }
  ],
  "gender": "female",
  "birthDate": "1968-03-12"
}


In [9]:
# An Observation resource is FHIR's version of a lab result or vital sign.
# Notice the same LOINC code (4548-4) and the same 7.2% as in HL7 and CDA —
# the CODE is what makes the three formats talk about the same fact.
from fhir.resources.observation import Observation

a1c = Observation(
    id="obs-a1c-1",
    status="final",  # the result is complete, not a preliminary draft
    # code.coding is a list so one observation can be tagged with several systems
    code={"coding": [{"system": "http://loinc.org",
                       "code": "4548-4",
                       "display": "Hemoglobin A1c"}]},
    # subject.reference points at the Patient resource we built above
    subject={"reference": "Patient/P1001"},
    # valueQuantity holds the number, the unit, and the UCUM unit system
    valueQuantity={"value": 7.2, "unit": "%",
                   "system": "http://unitsofmeasure.org", "code": "%"},
)
print(a1c.model_dump_json(indent=2))


{
  "resourceType": "Observation",
  "id": "obs-a1c-1",
  "status": "final",
  "code": {
    "coding": [
      {
        "system": "http://loinc.org",
        "code": "4548-4",
        "display": "Hemoglobin A1c"
      }
    ]
  },
  "subject": {
    "reference": "Patient/P1001"
  },
  "valueQuantity": {
    "value": 7.2,
    "unit": "%",
    "system": "http://unitsofmeasure.org",
    "code": "%"
  }
}


### Milestone 2 - bundle resources for exchange

In FHIR, a Bundle groups multiple resources together, allowing them to be exchanged as a single package between systems (for example, when sending the results of a batch query or submitting multiple records in one API call).

In [10]:
from fhir.resources.bundle import Bundle
bundle = Bundle(
    type="collection",
    entry=[
        {"resource": patient, "fullUrl": "Patient/P1001"},
        {"resource": a1c,     "fullUrl": "Observation/obs-a1c-1"},
    ],
)
print("Bundle type:", bundle.type)
print("Entries    :", len(bundle.entry))
print("First entry resourceType:", bundle.entry[0].resource.get_resource_type())
print("\nA FHIR Bundle is what a server returns from a query and what apps POST back.")

Bundle type: collection
Entries    : 2
First entry resourceType: Patient

A FHIR Bundle is what a server returns from a query and what apps POST back.


### Milestone 3 - the same fact in three standards

Here we illustrate how a single clinical fact, Ava Khan's A1c is 7.2%, can be represented using three different healthcare data standards: HL7 v2, CDA (an XML-based standard), and FHIR (a modern JSON-based standard). Although each format has its own structure, they all use the same standardized LOINC code (4548-4) and value to represent the fact, enabling interoperability across systems.


In [11]:
# Same clinical fact — Ava Khan's A1c is 7.2% — in three packaging standards.
# The LOINC code 4548-4 is the glue; only the envelope changes.
print("HL7 v2 (OBX):")
print("  OBX|1|NM|4548-4^Hemoglobin A1c^LN||7.2|%|4.0-5.6|H")
print("\nCDA (XML):")
print('  <observation><code code="4548-4" .../><value unit="%" value="7.2"/></observation>')
print("\nFHIR (JSON):")
print('  {"code":{"coding":[{"system":"http://loinc.org","code":"4548-4"}]},')
print('   "valueQuantity":{"value":7.2,"unit":"%"}}')
print("\nSame LOINC 4548-4, same 7.2% — three packaging standards.")


HL7 v2 (OBX):
  OBX|1|NM|4548-4^Hemoglobin A1c^LN||7.2|%|4.0-5.6|H

CDA (XML):
  <observation><code code="4548-4" .../><value unit="%" value="7.2"/></observation>

FHIR (JSON):
  {"code":{"coding":[{"system":"http://loinc.org","code":"4548-4"}]},
   "valueQuantity":{"value":7.2,"unit":"%"}}

Same LOINC 4548-4, same 7.2% — three packaging standards.


## Exercises

1. Add a second `OBX` (heart rate, LOINC `8867-4`) to the HL7 message and confirm
   your parser picks it up without code changes.
2. Extend the CDA fragment with a second `<observation>` and adjust the XPath to
   return a list of all observations.
3. Build a FHIR `Condition` resource for hypertension (SNOMED `38341003`) and add
   it to the bundle.



## Key takeaways

- Formats climb a ladder: **flat file → HL7 v2 → CDA → FHIR**, each trading
  simplicity for richer, more interoperable semantics.
- **HL7 v2** still dominates live messaging; **FHIR** is where new work happens —
  learn both.
- The *same coded fact* travels across all of them; **codes (Module 3) are the
  glue** that makes exchange meaningful.



---
*End of Part I — Foundations. Next: Part II — Interoperability and Data Engineering.*
